# PolypDB — Multi-Model Comparison
**YOLOv8n vs YOLOv11n vs YOLOv11s**

Based on พี่ฟิล์ม's pipeline + improvements:
- HSV augmentation enabled (color-space diversity across modalities)
- 100 epochs (vs 50)
- 3 model comparison

**Before running:**
Add dataset in Kaggle UI → Add Data → search `polypdb` by `debeshjha1`
Dataset will be at `/kaggle/input/polypdb/`

In [ ]:
# 1. Install
!pip install ultralytics opencv-python pandas numpy pyyaml tqdm -q

In [ ]:
# 2. Clone PolypDB repo (need split CSV files)
!git clone https://github.com/DebeshJha/PolypDB.git -q
print('Cloned PolypDB repo')

In [ ]:
# 3. Check GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 4. Prepare dataset — mask PNG → YOLO bbox
# (พี่ฟิล์ม's proven pipeline)
import cv2
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np

ROOT_DIR     = Path('PolypDB')
MODALITY_DIR = Path('/kaggle/input/polypdb/PolypDB/PolypDB/PolypDB_modality_wise')
SPLIT_DIR    = ROOT_DIR / 'dataset-split/Modality-Wise'
OUTPUT_DIR   = Path('/kaggle/working/yolo_dataset')

for split in ['train', 'valid', 'test']:
    (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def build_file_index(folder):
    return {f.stem: f for f in folder.iterdir() if f.is_file()}

def mask_to_yolo_bbox(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    h, w = mask.shape
    bboxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        x, y, wb, hb = cv2.boundingRect(cnt)
        xc = (x + wb/2.0) / w
        yc = (y + hb/2.0) / h
        bboxes.append(f'0 {xc:.6f} {yc:.6f} {wb/w:.6f} {hb/h:.6f}')
    return '\n'.join(bboxes) if bboxes else None

modalities = ['BLI', 'FICE', 'LCI', 'NBI', 'WLI']
found = missing = empty = 0

for mod in modalities:
    print(f'Processing {mod}...')
    img_index  = build_file_index(MODALITY_DIR / mod / 'images')
    mask_index = build_file_index(MODALITY_DIR / mod / 'masks')
    split_path = SPLIT_DIR / mod

    for split in ['train', 'valid', 'test']:
        csv_file = split_path / f'{split}.csv'
        if not csv_file.exists():
            print(f'  Missing CSV: {csv_file}')
            continue
        df = pd.read_csv(csv_file)
        for row in tqdm(df.itertuples(index=False), total=len(df), desc=f'{mod}-{split}', leave=False):
            img_stem  = Path(str(row.ImagePath)).stem
            mask_stem = Path(str(row.MaskPath)).stem
            src_img   = img_index.get(img_stem)
            src_mask  = mask_index.get(mask_stem)
            if src_img is None or src_mask is None:
                missing += 1
                continue
            unique_name = f'{mod}_{img_stem}'
            shutil.copy2(src_img, OUTPUT_DIR / 'images' / split / f'{unique_name}{src_img.suffix}')
            bbox_data = mask_to_yolo_bbox(src_mask)
            label_path = OUTPUT_DIR / 'labels' / split / f'{unique_name}.txt'
            with open(label_path, 'w') as lf:
                if bbox_data:
                    lf.write(bbox_data)
                else:
                    empty += 1
            found += 1

print(f'\nDone: {found:,} images | missing: {missing} | empty masks: {empty}')
for split in ['train', 'valid', 'test']:
    n = len(list((OUTPUT_DIR / 'images' / split).glob('*')))
    print(f'  {split.upper()}: {n:,} images')

In [ ]:
# 5. Write dataset YAML
import yaml
yaml_path = Path('/kaggle/working/polyp.yaml')
yaml_data = {
    'path': str(OUTPUT_DIR),
    'train': 'images/train',
    'val':   'images/valid',
    'test':  'images/test',
    'nc': 1,
    'names': ['Polyp']
}
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)
print('polyp.yaml ready')

In [ ]:
# 6. Shared augmentation config
AUG = dict(
    fliplr=0.5,
    flipud=0.5,
    degrees=15.0,
    scale=0.5,
    mixup=0.1,
    # HSV enabled (พี่ฟิล์มปิดไว้ — เปิดเพื่อช่วย cross-modality)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

TRAIN_COMMON = dict(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    seed=42,
    deterministic=True,
    plots=True,
    save=True,
    **AUG
)
print('Config ready')

In [ ]:
# 7. Train YOLOv8n
from ultralytics import YOLO
model_v8n = YOLO('yolov8n.pt')
model_v8n.train(
    project='/kaggle/working/runs',
    name='v8n',
    **TRAIN_COMMON
)
print('YOLOv8n done')

In [ ]:
# 8. Train YOLOv11n
model_v11n = YOLO('yolo11n.pt')
model_v11n.train(
    project='/kaggle/working/runs',
    name='v11n',
    **TRAIN_COMMON
)
print('YOLOv11n done')

In [ ]:
# 9. Train YOLOv11s
model_v11s = YOLO('yolo11s.pt')
model_v11s.train(
    project='/kaggle/working/runs',
    name='v11s',
    **TRAIN_COMMON
)
print('YOLOv11s done')

In [ ]:
# 10. Evaluate all 3 models — per-modality
import os
import shutil
import numpy as np
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

MODELS = {
    'YOLOv8n':  '/kaggle/working/runs/v8n/weights/best.pt',
    'YOLOv11n': '/kaggle/working/runs/v11n/weights/best.pt',
    'YOLOv11s': '/kaggle/working/runs/v11s/weights/best.pt',
}

def eval_per_modality(model_path, model_name):
    model = YOLO(model_path)
    modalities = ['BLI', 'FICE', 'LCI', 'NBI', 'WLI']
    rows = []

    for mod in modalities:
        temp_dir = Path(f'/kaggle/working/tmp_{mod}')
        (temp_dir / 'images').mkdir(parents=True, exist_ok=True)
        (temp_dir / 'labels').mkdir(parents=True, exist_ok=True)

        test_labels = list((OUTPUT_DIR / 'labels' / 'test').glob(f'{mod}_*.txt'))
        copied = 0
        for lbl in test_labels:
            for ext in ['.jpg', '.png', '.jpeg']:
                img = OUTPUT_DIR / 'images' / 'test' / f'{lbl.stem}{ext}'
                if img.exists():
                    shutil.copy2(img, temp_dir / 'images' / img.name)
                    shutil.copy2(lbl, temp_dir / 'labels' / lbl.name)
                    copied += 1
                    break
        if copied == 0:
            continue

        tmp_yaml = f'/kaggle/working/eval_{mod}.yaml'
        with open(tmp_yaml, 'w') as f:
            f.write(f'path: {temp_dir}\ntrain: images\nval: images\ntest: images\nnc: 1\nnames: ["Polyp"]\n')

        m = model.val(data=tmp_yaml, split='test', plots=False, verbose=False)
        P  = m.results_dict.get('metrics/precision(B)', 0)
        R  = m.results_dict.get('metrics/recall(B)', 0)
        m50 = m.results_dict.get('metrics/mAP50(B)', 0)
        F1 = (2*P*R)/(P+R) if (P+R) > 0 else 0

        rows.append({'Model': model_name, 'Modality': mod,
                     'Images': copied, 'Precision': round(P,4),
                     'Recall': round(R,4), 'F1': round(F1,4), 'mAP@0.5': round(m50,4)})

        shutil.rmtree(temp_dir)
        Path(tmp_yaml).unlink(missing_ok=True)

    return rows

all_results = []
for name, path in MODELS.items():
    if not Path(path).exists():
        print(f'Skipping {name} — weights not found')
        continue
    print(f'\nEvaluating {name}...')
    all_results.extend(eval_per_modality(path, name))

df = pd.DataFrame(all_results)
print(df.to_string(index=False))

In [ ]:
# 11. Comparison table — WLI focus + vs พี่ฟิล์ม baseline
BASELINE = {
    'Model': 'YOLOv11s (พี่ฟิล์ม · 50ep · no HSV)',
    'WLI_mAP50': 0.9409, 'WLI_Recall': 0.8429, 'WLI_F1': 0.8973
}

print('='*70)
print('COMPARISON TABLE — WLI (main modality)')
print('='*70)
print(f'{"Model":<35} {"mAP@0.5":>9} {"Recall":>8} {"F1":>8}')
print('-'*70)
print(f'{BASELINE["Model"]:<35} {BASELINE["WLI_mAP50"]:>9.4f} {BASELINE["WLI_Recall"]:>8.4f} {BASELINE["WLI_F1"]:>8.4f}')

wli_df = df[df['Modality'] == 'WLI']
for _, row in wli_df.iterrows():
    label = f"{row['Model']} (100ep + HSV)"
    print(f'{label:<35} {row["mAP@0.5"]:>9.4f} {row["Recall"]:>8.4f} {row["F1"]:>8.4f}')
print('='*70)

print('\nFull results:')
print(df.pivot_table(index='Modality', columns='Model', values='mAP@0.5').round(4).to_string())

In [ ]:
# 12. Model size + inference speed comparison
import time
results_size = []

for name, path in MODELS.items():
    if not Path(path).exists():
        continue
    model = YOLO(path)
    pt_mb = os.path.getsize(path) / 1e6

    dummy = torch.zeros(1, 3, 640, 640).cuda()
    for _ in range(5):  # warmup
        model.predict(dummy, verbose=False, device=0)
    times = []
    for _ in range(30):
        t0 = time.perf_counter()
        model.predict(dummy, verbose=False, device=0)
        times.append((time.perf_counter()-t0)*1000)
    avg_ms = round(np.mean(times), 1)

    results_size.append({'Model': name, 'Size (MB)': round(pt_mb,1), 'Inference (ms)': avg_ms, 'FPS': round(1000/avg_ms)})

df_size = pd.DataFrame(results_size)
print('\n' + '='*50)
print('MODEL SIZE & SPEED')
print('='*50)
print(df_size.to_string(index=False))
print('='*50)

In [ ]:
# 13. Save best model weights for download
# Download these files before closing session!
import shutil

for name, path in MODELS.items():
    if Path(path).exists():
        dest = f'/kaggle/working/DOWNLOAD_{name}_best.pt'
        shutil.copy2(path, dest)
        print(f'Saved: {dest} ({os.path.getsize(dest)/1e6:.1f} MB)')

print('\nDone! Download files above before closing session.')